In [ ]:
!pip install -q optuna

In [ ]:
import os
import torchvision
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from PIL import Image
import copy
import time
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
import pandas as pd
import numpy as np
import optuna
import shutil
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import classification_report
import optuna
import pandas as pd
import numpy as np
import os
import warnings
import tempfile
from optuna.pruners import SuccessiveHalvingPruner

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
class FishEyes(Dataset):
    def __init__(self, data, transform=None, use_cache=True):
        self.paths = data["Path"].tolist()
        self.labels = data["Label"].tolist()
        self.transform = transform
        self.use_cache = use_cache

        self.cache =[]

        resize_transform = transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR)

        if self.use_cache:
            print(f"Loading {len(self.paths)} images into RAM. This process may take 1-2 minutes...")
            for path in self.paths:
                img = Image.open(path).convert("RGB")
                img = resize_transform(img)
                self.cache.append(img)
            print("Finished loading data into RAM!")

    def __getitem__(self, index):
        label = torch.tensor(self.labels[index]).long()

        if self.use_cache:
            img = self.cache[index]
        else:
            path = self.paths[index]
            img = Image.open(path).convert("RGB")
            img = transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR)(img)

        if self.transform:
            img = self.transform(img)

        return img, label

    def __len__(self):
        return len(self.labels)

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

def get_dataloaders(batch_size, df_train, df_val, df_test):

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    transform = {
        "Train": transforms.Compose([  
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=20),
            transforms.ColorJitter(brightness=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        
        "Validation": transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        
        "Test": transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
    }

    train_dataset = FishEyes(data=df_train, transform=transform["Train"], use_cache=True)
    valid_dataset = FishEyes(data=df_val, transform=transform["Validation"], use_cache=True)
    test_dataset = FishEyes(data=df_test, transform=transform["Test"], use_cache=True)
    
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(dataset=valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)
    
    return train_loader, val_loader, test_loader


In [ ]:
df = pd.read_csv(r"meta_data.csv")

label_encoder = LabelEncoder()
df["Label"] = label_encoder.fit_transform(df["Label"])

df

In [ ]:
df_train = df.loc[df["Type"] == "Train"].copy().reset_index(drop=True)
df_val = df.loc[df["Type"] == "Validation"].copy().reset_index(drop=True)
df_test = df.loc[df["Type"] == "Test"].copy().reset_index(drop=True)

df_train.shape, df_val.shape, df_test.shape

In [ ]:
classes = df_train["Label"].unique()

print(classes)

In [ ]:
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_MODEL_PATH = "best_model.pt"

def build_model(dropout):
    model = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 3)
    )
    return model.to(device)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()


def evaluate(model, loader, criterion):
    model.eval()
    y_true, y_pred = [], []
    total_loss = 0.0  

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x) 
            loss = criterion(out, y)
            total_loss += loss.item()
            pred = out.argmax(dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    return acc, report, avg_loss

def objective(trial):

    start_time = time.time()

    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])
    lr = trial.suggest_float("lr", 5e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    MAX_EPOCHS = 100
    
    train_loader, val_loader, test_loader = get_dataloaders(batch_size, df_train, df_val, df_test)
    model = build_model(dropout)
    criterion = nn.CrossEntropyLoss()
    
    if optimizer_name == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"trial_{trial.number}.pt")

    best_val_acc = 0.0

    for epoch in range(MAX_EPOCHS):

        train_one_epoch(model, train_loader, optimizer, criterion)

        val_acc, report, val_loss = evaluate(model, val_loader, criterion)

        print(
            f"Trial {trial.number} | "
            f"Epoch {epoch+1}/{MAX_EPOCHS} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Best Val Acc: {best_val_acc:.4f}"
            )
        
        trial.report(val_acc, step=epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(model.state_dict(), checkpoint_path)

            trial.set_user_attr("checkpoint_path", checkpoint_path)
            trial.set_user_attr("best_epoch", epoch + 1)

    trial_time = time.time() - start_time

    trial.set_user_attr("params", trial.params)
    trial.set_user_attr("best_val_accuracy", best_val_acc)
    trial.set_user_attr("trial_time_sec", trial_time)

    del model
    del optimizer
    torch.cuda.empty_cache()
    
    return best_val_acc

def save_best_model_callback(study, trial):

    checkpoint_path = trial.user_attrs.get("checkpoint_path")

    if study.best_trial.number == trial.number:

        if checkpoint_path and os.path.exists(checkpoint_path):
            shutil.copyfile(checkpoint_path, BEST_MODEL_PATH)
            print(
                f"Trial {trial.number} is the best one yet. "
                f"Accuracy: {trial.value:.4f}. "
                f"Model saved to {BEST_MODEL_PATH}"
            )

    for f in os.listdir(CHECKPOINT_DIR):
        path = os.path.join(CHECKPOINT_DIR, f)

        if path != BEST_MODEL_PATH:
            os.remove(path)

In [ ]:
STUDY_NAME="Swin TPE + ASHA"
N_TRIALS = 20

if os.path.exists(BEST_MODEL_PATH):
    os.remove(BEST_MODEL_PATH)
    
study = optuna.create_study(
    direction="maximize",
    pruner=SuccessiveHalvingPruner(
        min_resource=20,            
        reduction_factor=3,      
        min_early_stopping_rate=0 
    ),
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name=STUDY_NAME
)


study.optimize(
    objective,
    n_trials=N_TRIALS,  
    show_progress_bar=True,
    callbacks=[save_best_model_callback]
)

In [ ]:
df_results = study.trials_dataframe()

cols_to_drop = [col for col in df_results.columns 
                if 'best_model_state_dict' in col or 'checkpoint_path' in col]

df_results.drop(columns=cols_to_drop, inplace=True, errors='ignore')

df_results.rename(columns={'value': 'best_val_accuracy'}, inplace=True)


best_trial = study.best_trial

best_model = build_model(best_trial.params["dropout"])
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
best_model.to(device)
best_model.eval()

_, _, test_loader = get_dataloaders(best_trial.params["batch_size"], df_train, df_val, df_test)
criterion = nn.CrossEntropyLoss()

test_acc, test_report, test_loss = evaluate(best_model, test_loader, criterion)

test_metrics = {
    "test_loss": test_loss,
    "test_accuracy": test_acc,
    "test_f1": test_report['weighted avg']['f1-score'],
    "test_precision": test_report['weighted avg']['precision'],
    "test_recall": test_report['weighted avg']['recall'],
}

for key, value in test_metrics.items():
    df_results.loc[df_results.number == best_trial.number, key] = value


rename_dict = {}
for col in df_results.columns:
    if col.startswith("user_attrs_"):
        rename_dict[col] = col.replace("user_attrs_", "")
    elif col.startswith("params_"):
        rename_dict[col] = col.replace("params_", "")

df_results.rename(columns=rename_dict, inplace=True)


info_cols = ['number', 'state', 'best_val_accuracy', 'best_epoch']
val_cols = sorted([col for col in df_results.columns if col.startswith('val_')])
test_cols = sorted(test_metrics.keys())
param_cols = sorted(best_trial.params.keys())
time_cols = ['duration', 'datetime_start', 'datetime_complete']


final_columns_order = info_cols + val_cols + test_cols + param_cols + time_cols
final_columns_order = [col for col in final_columns_order if col in df_results.columns]

df_results_final = df_results[final_columns_order]

csv_filename = f"{STUDY_NAME}_results.csv"
df_results_final.to_csv(csv_filename, index=False)

In [ ]:
print("\n===== Best Trial =====")
print(f"Trial number: {best_trial.number}")
print(f"Best validation accuracy: {best_trial.value:.4f}")

print("\nBest hyperparameters:")
for k, v in best_trial.params.items():
    print(f"{k}: {v}")

print("\n===== Test Set Performance =====")
print(f"Test Loss: {test_metrics['test_loss']:.4f}")
print(f"Test Accuracy: {test_metrics['test_accuracy']:.4f}")
print(f"Test F1-score: {test_metrics['test_f1']:.4f}")
print(f"Test Precision: {test_metrics['test_precision']:.4f}")
print(f"Test Recall: {test_metrics['test_recall']:.4f}")